In [14]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sns.set_theme(style="darkgrid", palette="muted", font_scale=1.1)

plt.rcParams.update({
    "figure.dpi":120
})

OUTPUT_DIR = Path("eda_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

ACCENT = "#58a6ff"
ACCENT2 = "#f78166"
ACCENT3 = "#3fb950"

In [15]:
CSV_PATH = r"C:\Users\Nived\Downloads\archive (7)\repositories.csv"

df = pd.read_csv(
    CSV_PATH,
    encoding="latin1",   
    low_memory=False
)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
df.head()

Rows: 215048
Columns: 1867


,Name,Description,URL,Created At,Updated At,Homepage,Size,Stars,Forks,Issues,...,Unnamed: 1857,Unnamed: 1858,Unnamed: 1859,Unnamed: 1860,Unnamed: 1861,Unnamed: 1862,Unnamed: 1863,Unnamed: 1864,Unnamed: 1865,Unnamed: 1866
0,freeCodeCamp,freeCodeCamp.org's open-source codebase and cu...,https://github.com/freeCodeCamp/freeCodeCamp,2014-12-24T17:49:19Z,2023-09-21T11:32:33Z,http://contribute.freecodecamp.org/,387451,374074,33599,248,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,free-programming-books,:books: Freely available programming books,https://github.com/EbookFoundation/free-progra...,2013-10-11T06:50:37Z,2023-09-21T11:09:25Z,https://ebookfoundation.github.io/free-program...,17087,298393,57194,46,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,awesome,ð Awesome lists about all kinds of interest...,https://github.com/sindresorhus/awesome,2014-07-11T13:42:37Z,2023-09-21T11:18:22Z,NaN,1441,269997,26485,61,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,996.ICU,Repo for counting stars and contributing. Pres...,https://github.com/996icu/996.ICU,2019-03-26T07:31:14Z,2023-09-21T08:09:01Z,https://996.icu,187799,267901,21497,16712,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,coding-interview-university,A complete computer science study plan to beco...,https://github.com/jwasham/coding-interview-un...,2016-06-06T02:34:12Z,2023-09-21T10:54:48Z,NaN,20998,265161,69434,56,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 215048 entries, 0 to 215047
Columns: 1867 entries, Name to Unnamed: 1866
dtypes: float64(8), object(1859)
memory usage: 3.0+ GB


In [17]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Unnamed: 1848,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Unnamed: 1849,1.0,59.0,NaN,59.0,59.0,59.0,59.0,59.0
Unnamed: 1850,1.0,5280.0,NaN,5280.0,5280.0,5280.0,5280.0,5280.0
Unnamed: 1851,1.0,754.0,NaN,754.0,754.0,754.0,754.0,754.0
Unnamed: 1852,1.0,3.0,NaN,3.0,3.0,3.0,3.0,3.0
Unnamed: 1853,1.0,5280.0,NaN,5280.0,5280.0,5280.0,5280.0,5280.0
Unnamed: 1854,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Unnamed: 1855,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
df.isnull().sum().sort_values(ascending=False)

Unnamed: 1848    215048
Unnamed: 1854    215048
Unnamed: 1855    215048
Unnamed: 1863    215047
Unnamed: 1862    215047
                  ...  
Forks                17
Size                 17
Created At           17
URL                  17
Name                  2
Length: 1867, dtype: int64

In [19]:
# strip whitespace
str_cols = df.select_dtypes("object").columns
df[str_cols] = df[str_cols].apply(lambda c: c.str.strip())

# convert datetime
df["Created At"] = pd.to_datetime(df["Created At"], errors="coerce")
df["Updated At"] = pd.to_datetime(df["Updated At"], errors="coerce")

# remove duplicates
df = df.drop_duplicates(subset=["URL"])

# remove forks
df = df[df["Is Fork"] != True]

# fill missing
df["Language"] = df["Language"].fillna("Unknown")
df["License"] = df["License"].fillna("No License")

df.shape

AttributeError: Can only use .str accessor with string values!

In [ ]:
NOW = pd.Timestamp.utcnow()

df["Age_Days"] = (NOW - df["Created At"]).dt.days
df["Age_Years"] = df["Age_Days"] / 365

df["Days_Since_Update"] = (NOW - df["Updated At"]).dt.days

df["Stars_Per_Year"] = df["Stars"] / df["Age_Years"].replace(0,np.nan)

df["Star_Fork_Ratio"] = df["Stars"] / df["Forks"].replace(0,np.nan)

df["Popularity_Score"] = (
    np.log1p(df["Stars"])*0.5 +
    np.log1p(df["Forks"])*0.3 +
    np.log1p(df["Watchers"])*0.2
)

df.head()

In [ ]:
df.nlargest(10,"Stars")[["Name","Language","Stars","Forks"]]

In [ ]:
df.nlargest(10,"Forks")[["Name","Language","Forks","Stars"]]

In [ ]:
lang_counts = df["Language"].value_counts().head(15)

plt.figure(figsize=(10,6))
sns.barplot(y=lang_counts.index,x=lang_counts.values)
plt.title("Top Languages")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(np.log1p(df["Stars"]),bins=50,color=ACCENT)
plt.title("Stars Distribution (log scale)")
plt.xlabel("log(Stars)")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(np.log1p(df["Forks"]),bins=50,color=ACCENT2)
plt.title("Forks Distribution (log scale)")
plt.show()

In [ ]:
corr_cols = ["Stars","Forks","Issues","Watchers","Size","Age_Days","Stars_Per_Year"]

corr = df[corr_cols].corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr,annot=True,cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(np.log1p(df["Forks"]),np.log1p(df["Stars"]),alpha=0.4)
plt.xlabel("log(Forks)")
plt.ylabel("log(Stars)")
plt.title("Forks vs Stars")
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(df["Age_Days"],np.log1p(df["Stars"]),alpha=0.4,color=ACCENT3)
plt.xlabel("Repo Age (Days)")
plt.ylabel("log(Stars)")
plt.title("Age vs Popularity")
plt.show()

In [ ]:
top_langs = df["Language"].value_counts().head(8).index

subset = df[df["Language"].isin(top_langs)]

subset["log_stars"] = np.log1p(subset["Stars"])

plt.figure(figsize=(12,6))
sns.boxplot(data=subset,x="Language",y="log_stars")
plt.title("Stars Distribution by Language")
plt.show()

In [ ]:
df["Size_Bucket"] = pd.cut(
    df["Size"],
    bins=[0,100,1000,10000,100000,np.inf],
    labels=["Tiny","Small","Medium","Large","Huge"]
)

sns.barplot(x="Size_Bucket",y="Stars",data=df)
plt.title("Median Stars by Size Bucket")
plt.show()

In [ ]:
Q1 = df["Stars"].quantile(0.25)
Q3 = df["Stars"].quantile(0.75)

IQR = Q3-Q1
upper = Q3 + 3*IQR

outliers = df[df["Stars"]>upper]

outliers[["Name","Stars","Forks"]].head()

In [ ]:
fig = px.scatter(
    df,
    x="Forks",
    y="Stars",
    color="Language",
    size="Stars",
    hover_name="Name",
    log_x=True,
    log_y=True
)

fig.show()

In [ ]:
print("Stars ↔ Forks correlation:",df["Stars"].corr(df["Forks"]))

print("\nTop Languages by Stars")

df.groupby("Language")["Stars"].sum().sort_values(ascending=False).head(10)

In [ ]:
print("Total repositories:",len(df))
print("Median stars:",df["Stars"].median())
print("Median age:",df["Age_Years"].median())

print("Most popular language:",df["Language"].value_counts().idxmax())

print("Most starred repo:",df.loc[df["Stars"].idxmax(),"Name"])

In [ ]:
ERq